# 03 Train Video Model

Train a CNN-LSTM hybrid model for deepfake detection using a staged training approach.

### Key Features:
- **Path Stability**: Use `PROJECT_ROOT` to resolve all paths from the project base.
- **Config-Driven**: Parameters from `video_config.yaml`.
- **Hardened Discovery**: Robust scanning for DFDC dataset parts.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from pathlib import Path
import numpy as np
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split
import os, cv2, json, yaml, logging
from PIL import Image

torch.cuda.empty_cache()

PROJECT_ROOT = Path().resolve().parents[1]

CONFIG_PATH = PROJECT_ROOT / "configs" / "video_config.yaml"
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

DATA_PATH = PROJECT_ROOT / config["data"]["raw_path"]
MODEL_PATH = PROJECT_ROOT / config["data"]["model_save_path"]

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

IMG_SIZE = config["model"]["frame_size"]
FRAME_COUNT = config["model"]["frame_count"]
BATCH_SIZE = config["training"]["batch_size"]
TRAIN_SUBSET = config["data"]["train_subset_size"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


## 1. Hardened Data Discovery

In [2]:
def extract_part_number(name):
    try:
        return int(name.split('_')[-1].strip())
    except:
        return None

def discover_dataset_paths(base_path, max_part=4):
    valid_parts = []

    for item in os.listdir(base_path):
        p = Path(base_path) / item

        if item.startswith("dfdc_train_part_") and p.is_dir():
            num = extract_part_number(item)
            if num is not None and num <= max_part:
                if (p / "metadata.json").exists():
                    valid_parts.append((p, num))

    valid_parts.sort(key=lambda x: x[1])
    print("Using parts:", [x[1] for x in valid_parts])

    return [x[0] for x in valid_parts]

## 2. Data Pipeline Execution

In [3]:
def get_video_list(parts):
    videos = []

    for part in parts:
        with open(part / "metadata.json") as f:
            meta = json.load(f)

        for file, info in meta.items():
            path = part / file
            if path.exists():
                label = 1 if info["label"] == "FAKE" else 0
                videos.append((str(path), label))

    return videos


class VideoFrameDataset(Dataset):
    def __init__(self, videos, num_frames, transform):
        self.videos = videos
        self.num_frames = num_frames
        self.transform = transform

    def __len__(self):
        return len(self.videos)

    def __getitem__(self, idx):
        path, label = self.videos[idx]

        cap = cv2.VideoCapture(path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        indices = np.linspace(0, total-1, self.num_frames).astype(int)
        frames = []

        for i in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, i)
            ret, frame = cap.read()

            if not ret:
                frame = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)

            frame = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            frame = self.transform(frame)
            frames.append(frame)

        cap.release()

        return torch.stack(frames), torch.tensor(label, dtype=torch.float32)


def build_dataloaders():
    parts = discover_dataset_paths(DATA_PATH, max_part=4)
    videos = get_video_list(parts)

    videos = videos[:TRAIN_SUBSET]

    train_v, val_v = train_test_split(videos, test_size=0.2, random_state=42)

    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])

    train_loader = DataLoader(
        VideoFrameDataset(train_v, FRAME_COUNT, transform),
        batch_size=BATCH_SIZE,
        shuffle=True,
        pin_memory=True,
        num_workers=0
    )

    val_loader = DataLoader(
        VideoFrameDataset(val_v, FRAME_COUNT, transform),
        batch_size=BATCH_SIZE,
        shuffle=False,
        pin_memory=True,
        num_workers=0
    )

    return train_loader, val_loader


train_loader, val_loader = build_dataloaders()
print("Train batches:", len(train_loader))

Using parts: [0, 1, 2, 3, 4]
Train batches: 2400


## 3. Model Architecture (CNN-LSTM)

In [4]:
class DeepfakeVideoModel(nn.Module):
    def __init__(self, hidden_dim=256, dropout=0.3):
        super().__init__()

        backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.feature_extractor = nn.Sequential(*list(backbone.children())[:-1])

        self.lstm = nn.LSTM(1280, hidden_dim, batch_first=True)

        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        B,T,C,H,W = x.shape
        x = x.view(B*T, C, H, W)

        f = self.feature_extractor(x).view(B*T, -1)
        f = f.view(B, T, -1)

        _, (h, _) = self.lstm(f)

        return self.fc(h[-1])


model = DeepfakeVideoModel().to(device)
criterion = nn.BCEWithLogitsLoss()
best_auc = 0.0

print("Model on:", next(model.parameters()).device)

Model on: cuda:0


## 4. Training Logic

In [ ]:
def train_epoch(model, loader, optimizer):
    model.train()
    total = 0

    scaler = torch.cuda.amp.GradScaler()  

    for x,y in tqdm(loader):
        x = x.to(device)
        y = y.to(device).unsqueeze(1)

        optimizer.zero_grad(set_to_none=True)
        
        with torch.cuda.amp.autocast():
            out = model(x)
            loss = criterion(out, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total += loss.item()

    return total/len(loader)


def validate(model, loader):
    model.eval()
    preds, labels = [], []

    with torch.no_grad():
        for x,y in loader:
            x = x.to(device)
            y = y.to(device)

            p = torch.sigmoid(model(x))

            preds.extend(p.cpu().numpy())
            labels.extend(y.cpu().numpy())

    return roc_auc_score(labels, preds)

## 5. Main Loop

In [6]:
optimizer = optim.Adam(model.parameters(), lr=0.0001)

for epoch in range(3):  # test run first
    loss = train_epoch(model, train_loader, optimizer)
    auc = validate(model, val_loader)

    print(f"Epoch {epoch+1} | Loss {loss:.4f} | AUC {auc:.4f}")

    if auc > best_auc:
        best_auc = auc
        torch.save(model.state_dict(), MODEL_PATH)

C:\Users\Asus\AppData\Local\Temp\ipykernel_22676\4024755675.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()  # 🔥 add here


  0%|          | 0/2400 [00:00<?, ?it/s]

C:\Users\Asus\AppData\Local\Temp\ipykernel_22676\4024755675.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1 | Loss 0.2407 | AUC 0.5376


C:\Users\Asus\AppData\Local\Temp\ipykernel_22676\4024755675.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()  # 🔥 add here


...

Epoch 2 | Loss 0.2344 | AUC 0.6104


C:\Users\Asus\AppData\Local\Temp\ipykernel_22676\4024755675.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()  # 🔥 add here


  0%|          | 0/2400 [00:00<?, ?it/s]

C:\Users\Asus\AppData\Local\Temp\ipykernel_22676\4024755675.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 3 | Loss 0.2337 | AUC 0.6337
